# Assignment 4, Problem 1: A linear finite element code for a bar

ME/BME 736 — Advanced Finite Element Methods (60 marks)

**Before you start.** Save a copy of this notebook as `lastname_assignment04_1d.ipynb` and select the kernel **Python (Finite Elements)**. Before you submit, run **Restart Kernel and Run All Cells** and check that every cell runs without an error.

Complete every cell marked **TODO**. You may add cells.

**Rules.** Use NumPy for the arrays and the linear solve and Matplotlib for the figures. Do not use FEniCSx or another finite element package. Chapter 5 and the lecture notebook `notebooks/assembly/assembly_demo.ipynb` cover every step.

## The problem

A uniform bar occupies $0<x<1$ with $AE=1$. It is fixed at $x=0$, carries a distributed axial load $f(x)$, and carries a point load $P$ at $x=1$:

$$
-u''(x)=f(x)\quad\text{for }0<x<1,
\qquad
u(0)=0,
\qquad
u'(1)=P.
$$

With linear basis functions $\phi_i$, the finite element equations use

$$
K_{ij}=\int_0^1\phi_j'(x)\,\phi_i'(x)\,\mathrm{d}x,
\qquad
F_i=\int_0^1 f(x)\,\phi_i(x)\,\mathrm{d}x+P\,\phi_i(1).
$$

Solve both cases below with the same code.

| case | exact displacement $u_\star(x)$ | distributed load $f(x)$ | point load $P$ |
|---|---:|---:|---:|
| A | $2x-x^2/2$ | $1$ | $1$ |
| B | $2x-x^3/3$ | $2x$ | $1$ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)  # print arrays with 6 digits

# Problem data. CASES["A"] holds the data for case A, and CASES["B"] for case B.
# "lambda x: ..." is a short way to write a one-line function of x.
#   CASES["A"]["f"](x)    distributed load f at x (x can be a number or an array)
#   CASES["A"]["u"](x)    exact displacement at x
#   CASES["A"]["du"](x)   derivative of the exact displacement at x
#   CASES["A"]["P"]       point load P (a number)
CASES = {
    "A": {"u": lambda x: 2 * x - x**2 / 2, "du": lambda x: 2 - x,
          "f": lambda x: 1.0 + 0.0 * x, "P": 1.0},  # 1 + 0*x has the same shape as x
    "B": {"u": lambda x: 2 * x - x**3 / 3, "du": lambda x: 2 - x**2,
          "f": lambda x: 2 * x, "P": 1.0},
}
MESHES = [4, 8, 16]  # numbers of elements

## Plotting helpers

Run this cell once. It is not part of the finite element algorithm.

In [ ]:
def plot_solutions(case_name, results, u_exact):
    # results: list of (N, nodes, U) tuples, one per mesh.
    x = np.linspace(0.0, 1.0, 400)  # 400 equally spaced points from 0 to 1
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.plot(x, u_exact(x), "k--", linewidth=2, label=r"exact $u_\star$")
    for N, nodes, U in results:
        ax.plot(nodes, U, "o-", markersize=4, label=rf"$N={N}$")
    ax.set(xlabel=r"$x$", ylabel=r"$u$", title=f"Case {case_name}: displacement")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()


def plot_errors(case_name, h, error_L2, error_H1):
    h = np.asarray(h)  # turn a list into a NumPy array
    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    # loglog: plot with logarithmic scales on both axes
    ax.loglog(h, error_L2, "o-", label=r"$\|u_\star-u_h\|_{L^2}$")
    ax.loglog(h, error_H1, "s-", label=r"$\|u_\star-u_h\|_{H^1}$")
    ax.set(xlabel=r"element size $h$", ylabel="error", title=f"Case {case_name}: errors")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

## Step 1. Mesh and connectivity (TODO)

Write `make_mesh(N)` for $N$ uniform elements. It returns the node coordinates and an $N\times 2$ integer array whose row $e$ holds the global node numbers of element $e$. Python counts from 0, so node 1 in the book is index 0 here.

In [ ]:
def make_mesh(N):
    # TODO: return nodes (length N+1) and connectivity (shape N x 2).
    raise NotImplementedError

## Step 2. Gauss quadrature (TODO)

Gauss quadrature replaces an integral on the reference interval by a weighted sum of function values:

$$
\int_{-1}^{1} g(\xi)\,\mathrm{d}\xi\approx\sum_{q=1}^{n} w_q\,g(\xi_q).
$$

With $n$ points, the sum is exact when $g$ is a polynomial of degree $2n-1$ or lower. The points $\xi_q$ and weights $w_q$ are:

| $n$ | points $\xi_q$ | weights $w_q$ |
|---|---|---|
| 2 | $\pm 1/\sqrt3\approx\pm 0.5773502691896258$ | $1$ |
| 4 | $\pm 0.3399810435848563$ | $0.6521451548625461$ |
|   | $\pm 0.8611363115940526$ | $0.3478548451374538$ |

Write `gauss_rule(n)` for $n=2$ and $n=4$. It returns the points and the weights as two NumPy arrays. Use $n=2$ for the element load vector and $n=4$ for the error norms.

In [ ]:
def gauss_rule(n):
    # TODO: return (points, weights) as NumPy arrays, for n = 2 and n = 4.
    raise NotImplementedError


# Check: both rules should print 0.666667, the exact value of the integral of xi^2 over [-1, 1].
for n in (2, 4):
    xi, w = gauss_rule(n)
    print(n, np.sum(w * xi**2))

## Step 3. Element matrix and load vector (TODO)

For element $\mathcal K_e=[x_a,x_b]$ of length $h_e$,

$$
\mathbf K^e=\frac{1}{h_e}\begin{bmatrix}1&-1\\-1&1\end{bmatrix},
\qquad
F_a^e=\int_{\mathcal K_e} f(x)\,\phi_a^e(x)\,\mathrm{d}x,\quad a=1,2.
$$

Compute $F_a^e$ with two-point Gauss quadrature (`gauss_rule(2)`). On the reference interval $-1\le\xi\le 1$ the basis functions are $\hat\phi_1=(1-\xi)/2$ and $\hat\phi_2=(1+\xi)/2$. The map to the element is $x=\tfrac12(x_a+x_b)+\tfrac{h_e}{2}\xi$, so $\mathrm{d}x=\tfrac{h_e}{2}\,\mathrm{d}\xi$.

In [ ]:
def element_matrix(h_e):
    # TODO: return the 2 x 2 element stiffness matrix.
    raise NotImplementedError


def element_load(x_a, x_b, f):
    # TODO: return the 2-entry element load vector using two-point Gauss quadrature.
    raise NotImplementedError

## Step 4. Assemble and solve (TODO)

Write `solve_bar(N, f, P)`. It should:

1. loop over the elements and add each entry of $\mathbf K^e$ and $\mathbf F^e$ to the global $\mathbf K$ and $\mathbf F$, using the connectivity and explicit loops over the local indices $a$ and $b$;
2. add $P$ to the last entry of $\mathbf F$;
3. set $U_1=0$ and solve the reduced system for the remaining nodal values (`np.linalg.solve(A, b)` returns the solution of $\mathbf A\mathbf x=\mathbf b$); and
4. recover the reaction $R_1$ at $x=0$ as the first entry of $\mathbf K\mathbf U-\mathbf F$, using the full assembled $\mathbf K$ and $\mathbf F$.

In [ ]:
def solve_bar(N, f, P):
    # TODO: return nodes, connectivity, U, and the reaction R1.
    raise NotImplementedError

## Step 5. Error norms (TODO)

Write `error_norms(nodes, connectivity, U, u_exact, du_exact)` that returns

$$
\|u_\star-u_h\|_{L^2(0,1)}=\left[\int_0^1(u_\star-u_h)^2\,\mathrm{d}x\right]^{1/2},
\qquad
\|u_\star-u_h\|_{H^1(0,1)}=\left[\int_0^1\left((u_\star-u_h)^2+(u_\star'-u_h')^2\right)\mathrm{d}x\right]^{1/2}.
$$

Integrate element by element with four-point Gauss quadrature (`gauss_rule(4)`). On each element, compute $u_h$ and $u_h'$ from the two nodal values and the linear basis functions. Do not interpolate from a plotting grid.

In [ ]:
def error_norms(nodes, connectivity, U, u_exact, du_exact):
    # TODO: return the L2 error and the H1 error.
    raise NotImplementedError

## Step 6. Results for cases A and B (TODO)

For each case, solve with $N=4$, $8$, and $16$, and:

1. plot the exact displacement and the three finite element solutions on one figure (use `plot_solutions`);
2. print a table of $N$, $h$, the $L^2$ error, and the $H^1$ error, and plot both errors against $h$ (use `plot_errors`);
3. print the observed rate between consecutive meshes, $\log(e_N/e_{2N})/\log 2$, for both errors; and
4. print the reaction $R_1$ and the equilibrium residual $R_1+\int_0^1 f\,\mathrm{d}x+P$ for every mesh. The residual should be zero up to rounding error.

In [ ]:
for name, case in CASES.items():  # name is "A" or "B"; case is CASES[name]
    # TODO: solve on each mesh in MESHES, then produce the plots and printed tables above.
    pass

## Conclusions (TODO)

In a few sentences for each case, state the observed $L^2$ and $H^1$ rates and whether the equilibrium residual is zero up to rounding error.